# CLAUD·IA — Escuchar el modelo antes y después del fine-tuning

Las dos versiones reciben **las mismas grabaciones personales** y traducen del español al inglés. Tanto el modelo original como el ajustado utilizan la misma síntesis de voz. Las grabaciones personales se utilizan exclusivamente para esta evaluación.

[Documentación de SeamlessM4T v2](https://huggingface.co/docs/transformers/model_doc/seamless_m4t_v2)

## 1. Preparar Colab

Selecciona **T4 GPU** en los ajustes de ejecución. Ejecuta la instalación. Si es necesario reiniciar el kernel, hazlo y después importa `fine_tuning_pilote.zip` y las dos grabaciones desde el panel **Archivos**. No descomprimas el ZIP.

In [ ]:
%pip install "torch>=2.6" "transformers[sentencepiece]==5.22.1" "huggingface-hub>=1.32,<2"

## 2. Preparar los audios y leer los pesos ajustados

FFmpeg extrae el sonido de los archivos MP4 o WAV y lo convierte en memoria a mono de 16 kHz. `ZipFile` lee el informe y los pesos directamente desde el ZIP exportado por el notebook 03.

Si un archivo tiene otro nombre, cambia su valor en `examples`.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import torch
from safetensors.torch import load

if not torch.cuda.is_available():
    raise RuntimeError("Selecciona T4 GPU en Colab.")
if shutil.which("ffmpeg") is None:
    raise RuntimeError("Se necesita FFmpeg para preparar las grabaciones.")

model_id = "facebook/seamless-m4t-v2-large"
sample_rate = 16_000
target_language = "eng"
device = torch.device("cuda")
archive_path = Path("fine_tuning_pilote.zip")
examples = [
    ("gracias_por_venir.mp4", "Gracias por venir.", "Thank you for coming."),
    ("antes_de_las_ocho.mp4", "No puedo llegar antes de las ocho.", "I can’t arrive before eight."),
]

def read_recording(filename):
    if not Path(filename).is_file():
        raise FileNotFoundError(f"Importa {filename} desde el panel Archivos.")
    conversion = subprocess.run(
        ["ffmpeg", "-nostdin", "-v", "error", "-i", filename, "-vn", "-ac", "1",
         "-ar", str(sample_rate), "-f", "f32le", "pipe:1"],
        capture_output=True,
    )
    if conversion.returncode != 0:
        raise RuntimeError(conversion.stderr.decode("utf-8", errors="replace"))
    audio = np.frombuffer(conversion.stdout, dtype="<f4").copy()
    if audio.size == 0:
        raise ValueError(f"No hay sonido en {filename}.")
    return audio

if not archive_path.is_file():
    raise FileNotFoundError("Importa fine_tuning_pilote.zip desde el panel Archivos.")

with ZipFile(archive_path) as archive:
    report = json.loads(archive.read("rapport.json"))
    adjusted_weights = load(archive.read("poids_ajustes.safetensors"))

if (report["base_model"], report["source_language"], report["target_language"]) != (
    model_id, "spa", target_language
):
    raise ValueError("El archivo comprimido debe corresponder al fine-tuning de español → inglés del notebook 03.")

recordings = [(source, reference, read_recording(filename)) for filename, source, reference in examples]
print(torch.cuda.get_device_name(0))
print(f"{len(recordings)} enregistrements et {len(adjusted_weights)} tensores chargés.")

## 3. Cargar el traductor de voz

Cargamos el modelo de voz una sola vez y conservamos una copia de sus cuatro tensores originales. Los pesos ajustados se guardaron en FP32. Los convertimos a FP16, como los demás pesos del traductor en T4. El notebook comprueba que sigan existiendo diferencias después de la conversión.

In [ ]:
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToSpeech

processor = AutoProcessor.from_pretrained(model_id)
model = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
    model_id, dtype=torch.float16, use_safetensors=True
).to(device)
model.eval()

layer_index = len(model.text_decoder.layers) - 1
prefix = f"text_decoder.layers.{layer_index}.cross_attention."
expected_names = {
    f"{prefix}{projection}.{kind}"
    for projection in ("q_proj", "v_proj")
    for kind in ("weight", "bias")
}
if adjusted_weights.keys() != expected_names:
    raise ValueError("Les noms des poids ne correspondent pas aux quatre tensores du pilote.")

parameters = {name: model.get_parameter(name) for name in expected_names}
original_weights = {name: parameter.detach().cpu().clone() for name, parameter in parameters.items()}
for name, weights in adjusted_weights.items():
    if weights.shape != parameters[name].shape or not torch.isfinite(weights).all():
        raise ValueError(f"Tensor incompatible: {name}")

adjusted_weights = {
    name: weights.to(dtype=parameters[name].dtype) for name, weights in adjusted_weights.items()
}
if all(torch.equal(original_weights[name], weights) for name, weights in adjusted_weights.items()):
    raise ValueError("Los pesos ajustados son idénticos a los pesos originales en FP16.")

@torch.no_grad()
def set_weights(weights):
    for name, parameter in parameters.items():
        parameter.copy_(weights[name])

print("Modelo de voz cargado. Los pesos ajustados corresponden al decodificador.")

## 4. Comparar las traducciones

Para cada frase, escucha el original y después ambas traducciones. Comprueba el significado, la negación y los números. Dos formulaciones diferentes pueden ser correctas. Estas dos frases constituyen una primera prueba con nuestras grabaciones.

In [ ]:
from IPython.display import Audio, display

@torch.inference_mode()
def translate(audio):
    inputs = processor(audio=audio, sampling_rate=sample_rate, return_tensors="pt").to(device)
    inputs["input_features"] = inputs["input_features"].to(dtype=torch.float16)
    result = model.generate(
        **inputs, tgt_lang=target_language, return_intermediate_token_ids=True,
        text_do_sample=False, speech_do_sample=False,
    )
    text = processor.batch_decode(
        result.sequences.cpu(), skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    length = result.waveform_lengths.item()
    waveform = result.waveform.reshape(-1)[:length].float().cpu().numpy()
    return text, waveform

set_weights(original_weights)
before = [translate(audio) for _, _, audio in recordings]
try:
    set_weights(adjusted_weights)
    after = [translate(audio) for _, _, audio in recordings]
finally:
    set_weights(original_weights)

for (source, reference, audio), original, tuned in zip(recordings, before, after):
    print(f"Frase: {source}")
    print(f"Referencia: {reference}")
    print("Audio original")
    display(Audio(audio, rate=sample_rate))
    for label, (text, waveform) in (("Antes", original), ("Después", tuned)):
        print(f"{label} : {text}")
        display(Audio(waveform, rate=model.config.sampling_rate))

## 5. Comparar voz normal y voz baja

En una sesión que ya haya completado la sección 4, copia únicamente la siguiente celda debajo de los resultados. Reutiliza el modelo cargado.

Graba dos veces **la misma frase**, con el mismo micrófono, a la misma distancia y en el mismo lugar. Habla normalmente en el primer archivo y más bajo en el segundo. Importa ambos archivos MP4. El nivel RMS indicado describe la señal grabada y permite comprobar si el archivo de « voz baja » realmente tiene un nivel inferior.

| Frase que debes grabar | Traducción de referencia al inglés |
| --- | --- |
| No puedo llegar antes de las ocho. | I can’t arrive before eight. |
| Gracias por venir. | Thank you for coming. |
| Necesito dos boletos para mañana. | I need two tickets for tomorrow. |
| El tren sale a las siete y media. | The train leaves at half past seven. |
| La farmacia está al lado del banco. | The pharmacy is next to the bank. |
| No olvides llevar el pasaporte. | Don’t forget to bring your passport. |

Escucha las versiones antes y después del fine-tuning para cada grabación. Asigna **2** puntos si se conserva el significado, **1** si falta alguna información y **0** si cambia el significado o no se puede utilizar el audio. Una formulación diferente en inglés también puede ser correcta. Repite la celda con los nombres de archivo y la referencia de cada frase. Registra las puntuaciones por condición y por versión.

In [ ]:
reference = "I can’t arrive before eight."
voice_files = [
    ("Voz normal", "ocho_normal.mp4"),
    ("Voz baja", "ocho_baja.mp4"),
]
voice_recordings = [(label, read_recording(filename)) for label, filename in voice_files]

set_weights(original_weights)
voice_before = [translate(audio) for _, audio in voice_recordings]
try:
    set_weights(adjusted_weights)
    voice_after = [translate(audio) for _, audio in voice_recordings]
finally:
    set_weights(original_weights)

print(f"Referencia: {reference}")
for (condition, audio), before_result, after_result in zip(voice_recordings, voice_before, voice_after):
    rms = np.sqrt(np.mean(audio.astype(np.float64) ** 2))
    rms_dbfs = 20 * np.log10(max(rms, 1e-12))
    print(f"\n{condition} • nivel RMS: {rms_dbfs:.1f} dBFS")
    for label, (text, waveform) in (("Antes", before_result), ("Después", after_result)):
        print(f"{label} : {text}")
        display(Audio(waveform, rate=model.config.sampling_rate))